In [ ]:
!pip install gdown -q
!pip install split-folders
!pip install tensorflow==2.19.0
!pip install matplotlib==3.9.2
!pip install numpy==2.0
!pip install scipy==1.14.1
!pip install scikit-learn==1.5.2
!pip install -q "ml_dtypes>=0.5.0"

## Download the input data file

In [10]:
import os
import zipfile
import gdown

# Define dataset URL and paths
file_id = "1HY71FfYNGX1lBB9hnfUjL_h9GWLN_w2Q"
local_zip = "waste-dataset.zip"
extract_dir = "waste-dataset"

def download_dataset(file_id, output_file):
    """Download the dataset using wget in quiet mode."""
    print("Downloading the dataset...")
    gdown.download(id=file_id, output=output_file, quiet=False)
    print("Download completed.")

def extract_zip_in_chunks(zip_file, extract_to, batch_size=2000):
    """
    Extract a large zip file in chunks to avoid memory bottlenecks.
    Processes a specified number of files (batch_size) at a time.
    """
    print("Extracting the dataset in chunks...")
    os.makedirs(extract_to, exist_ok=True)  # Ensure the extraction directory exists
    
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        files = zip_ref.namelist()  # List all files in the archive
        total_files = len(files)
        
        for i in range(0, total_files, batch_size):
            batch = files[i:i+batch_size]
            for file in batch:
                zip_ref.extract(file, extract_to)  # Extract each file in the batch
            print(f"Extracted {min(i+batch_size, total_files)} of {total_files} files...")
    
    print(f"Dataset successfully extracted to '{extract_to}'.")

# Main script execution
if __name__ == "__main__":

    os.system("pip install gdown -q")

    # Download the dataset if not already downloaded
    if not os.path.exists(local_zip):
        download_dataset(file_id, local_zip)
    else:
        print("Dataset already downloaded.")
    
    # Extract the dataset if not already extracted
    if not os.path.exists(extract_dir):
        extract_zip_in_chunks(local_zip, extract_dir)
    else:
        print("Dataset already extracted.")
    
    # Optional cleanup of the zip file
    if os.path.exists(local_zip):
        os.remove(local_zip)
        print(f"Cleaned up zip file: {local_zip}") 

Downloading...
From (original): https://drive.google.com/uc?id=1HY71FfYNGX1lBB9hnfUjL_h9GWLN_w2Q
From (redirected): https://drive.google.com/uc?id=1HY71FfYNGX1lBB9hnfUjL_h9GWLN_w2Q&confirm=t&uuid=98856462-b164-40e9-8ae7-2170d2b0e56a
To: c:\Users\Jocel\Downloads\AI project\final-project-tensorflow\waste-dataset.zip
100%|██████████| 587M/587M [00:10<00:00, 53.6MB/s] 


Download completed.
Extracting the dataset in chunks...
Extracted 2000 of 5989 files...
Extracted 4000 of 5989 files...
Extracted 5989 of 5989 files...
Dataset successfully extracted to 'waste-dataset'.
Cleaned up zip file: waste-dataset.zip


## Task 1: Import necessary libraries and set dataset paths 

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import splitfolders
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout, BatchNormalization
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

# Print TensorFlow version
print(tf.__version__)

# Split 70% train, 15% val, 15% test
splitfolders.ratio("waste-dataset", output="waste-split", ratio=(0.7, 0.15, 0.15), seed=42)

# Set dataset paths
train_dir = 'waste-split/train'
val_dir   = 'waste-split/val'
test_dir  = 'waste-split/test'

2.16.2


Copying files: 0 files [00:00, ? files/s]


## Task 2: Set up data generators for training, validation, and testing with augmentation 

In [ ]:
# Image data generators
train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Print the length of the training data generator
print(len(train_datagen))

val_datagen = ImageDataGenerator(rescale=1.0/255.0)
test_datagen = ImageDataGenerator(rescale=1.0/255.0)

# Load images from directories
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(64, 64),
    batch_size=16,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(64, 64),
    batch_size=16,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(64, 64),
    batch_size=16,
    class_mode='categorical'
)

Found 0 images belonging to 4 classes.
Found 0 images belonging to 4 classes.
Found 0 images belonging to 4 classes.


## Task 3: Define the VGG16-based model architecture with custom layers

In [ ]:
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, BatchNormalization, Dropout

# Load VGG16 with pre-trained weights
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(64, 64, 3))

# Freeze the base model layers
for layer in base_model.layers:
    layer.trainable = False

# Build the model
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(train_generator.num_classes, activation='softmax')
])
